In [2]:
import numpy as np
import matplotlib.pyplot as plt
import nifty8 as ift
from phase_II.utils.helpers import Stress, visualize_stress, usual_plot
from phase_I.utils.config_jupyter_notebooks import *
%matplotlib tk

nrt_strain_values = np.loadtxt("../../data/data_txt/num_rel_template_strain_values.txt") * 1e19
nrt_time_values = np.loadtxt("../../data/data_txt/num_rel_template_time_values.txt") - zero_time

Important variables: 
		signal_strip_time, signal_strip_strain 
		signal_strip_strain_tapered
		strain
		time_domain_strip
		N


In [4]:
L = np.max(nrt_time_values) - np.min(nrt_time_values)
n = len(nrt_time_values)
time_space = ift.RGSpace((n,), distances=L/n)

nrt_field = ift.Field(ift.DomainTuple.make(time_space), val=nrt_strain_values)

_ = plt.figure(figsize=(10,6))
plt.plot(nrt_time_values, nrt_field.val)
usual_plot()

In [5]:
r_dom = nrt_field.domain
h_dom = r_dom[0].get_default_codomain()
F = ift.FFTOperator(domain=r_dom)
harmonic_nrt_field = F(nrt_field)

empirical_ps_nrt = ift.power_analyze(harmonic_nrt_field)

In [36]:
_ = plt.figure()
plt.plot(h_dom.get_unique_k_lengths(), empirical_ps_nrt.val)
plt.show()
plt.loglog()

[]

In [6]:
S_mat, t_dual, f = Stress(nrt_field)


Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Fourier-Transforming columns of Phi matrix
	 ... Done
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (1.480359213267309e-22) 


In [7]:
visualize_stress(S_mat, rows=f, cols=t_dual+min(nrt_time_values))

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


## Try to invert the Wigner function

In [8]:
t_vol = r_dom[0].scalar_dvol
N = r_dom[0].shape[0]
h_vol = h_dom.scalar_dvol*N

FFT_1 = ift.FFTOperator(domain=(h_dom, r_dom[0]), space=1) * (1/t_vol)

S_mat_field = ift.Field(domain=ift.DomainTuple.make((h_dom, r_dom[0])), val=S_mat)

In [9]:
Sigma_f_q = FFT_1(S_mat_field).val

In [12]:
plt.matshow(np.abs(Sigma_f_q))

In [219]:
# Get Sigma (q/2, q)
k_half = 0.5 * f.copy()
xi_0 = harmonic_nrt_field.val[0]

### Approach: Just pick out every second wavevector => Introduces shift and stretch in time
k_half_in_f = np.isin(k_half, f)
masked_k_half = k_half[k_half_in_f]

valid_columns = np.where(k_half_in_f)[0]

print(valid_columns)

valid_rows = np.array([np.where(f == kh)[0][0] for kh in masked_k_half])

# reconstructed_xi_tilde_phase = np.zeros(len(f), dtype=complex)
# to_insert = Sigma_f_q[valid_rows, valid_columns]
# reconstructed_xi_tilde_phase[valid_columns] = to_insert

reconstructed_xi_tilde_phase = Sigma_f_q[valid_rows, valid_columns]
reconstructed_xi_tilde_wo_phase = reconstructed_xi_tilde_phase/xi_0.conj()

delta_k = masked_k_half[1] - masked_k_half[0]
K = len(masked_k_half)
dual_coarse_time = np.arange(K) / (K * delta_k)
dual_fine_time = np.arange(len(nrt_time_values)) / (len(nrt_time_values) * (f[1]-f[0]))

coarser_h_dom = ift.RGSpace(harmonic=True, shape=(len(masked_k_half), ), distances=delta_k)

h_vol_coarser = coarser_h_dom.scalar_dvol*len(masked_k_half)
FFT_2_coarse = ift.FFTOperator(domain=coarser_h_dom) * (1/h_vol_coarser)

FFT_2_fine = ift.FFTOperator(domain=h_dom) * (1/h_vol)

reconstructed_xi_tilde_phase_field =  ift.Field(ift.DomainTuple.make(coarser_h_dom), val=reconstructed_xi_tilde_phase)
reconstructed_xi_tilde_wo_phase_field =  ift.Field(ift.DomainTuple.make(coarser_h_dom), val=reconstructed_xi_tilde_wo_phase)

### Approach: Linear interpolate

# preallocate
# reconstructed_xi_tilde_phase = np.empty_like(k_half, dtype=complex)

# interpolate each column of Sigma_f_q along the first axis
# for j in range(Sigma_f_q.shape[1]):
#     reconstructed_xi_tilde_phase[j] = np.interp(
#         np.fft.fftshift(k_half)[j],       # target x
#         np.fft.fftshift(f),               # original x, need to be ordered monotonically => np.fft.fftshift
#         np.fft.fftshift(Sigma_f_q[:, j].real)  # y-values at original x
#     )
#     +1j * np.interp(
#         np.fft.fftshift(k_half)[j],
#         np.fft.fftshift(f),
#         np.fft.fftshift(Sigma_f_q[:, j].imag)
#     )
#
# reconstructed_xi_tilde_phase = np.fft.ifftshift(reconstructed_xi_tilde_phase)
# reconstructed_xi_tilde_wo_phase = reconstructed_xi_tilde_phase/xi_0.conj()  # inverse shift to get ready for FFT!
#
# FFT_2 = ift.FFTOperator(domain=h_dom) * (1/h_vol)
#
# reconstructed_xi_tilde_phase_field =  ift.Field(ift.DomainTuple.make(h_dom), val=reconstructed_xi_tilde_phase)
# reconstructed_xi_tilde_wo_phase_field =  ift.Field(ift.DomainTuple.make(h_dom), val=reconstructed_xi_tilde_wo_phase)


[   0    2    4 ... 8187 8189 8191]


In [234]:
recovered_xi = FFT_2_coarse(reconstructed_xi_tilde_wo_phase_field).val
recovered_xi_normed = recovered_xi/np.max(recovered_xi) * np.max(nrt_field.val)

recovered_xi_up_to_phase = FFT_2_coarse(reconstructed_xi_tilde_phase_field).val
recovered_xi_up_to_phase_normed = recovered_xi_up_to_phase/np.max(recovered_xi_up_to_phase) * np.max(nrt_field.val)

In [236]:
# plt.plot(np.arange(len(recovered_xi_normed)), recovered_xi_up_to_phase_normed, label="Up to global phase")
plt.plot(np.arange(len(recovered_xi_normed)), recovered_xi_normed, label="Fourier back transform of inverse wigner mat (scaled f.v.p.!)")
plt.plot(np.arange(len(nrt_field.val))-4100, nrt_field.val, label="Original input field ")
usual_plot()

In [213]:
import numpy as np
import matplotlib.pyplot as plt

# original signal in frequency domain
N = 16
X = np.zeros(N, dtype=complex)
X[3] = 1  # single frequency component
X[-3] = 1  # symmetric component for real signal

# IFFT of original
x = np.fft.ifft(X, norm="forward")

# zero-inserted version (double length)
X_upsampled = np.zeros(2 * N, dtype=complex)
X_upsampled[::2] = X  # insert zeros between every freq bin

# IFFT of zero-inserted signal
x_upsampled = np.fft.ifft(X_upsampled, norm="forward")

t = np.arange(N) / (N * delta_k)
t_upsampled = np.arange(2*N) / (2 * N * delta_k)

print(max(t_upsampled), max(t))


# plot
plt.figure(figsize=(8, 4))
plt.plot(t_upsampled, x_upsampled.real, '-.', label='IFFT after zero insertion')
plt.plot(t, x.real, '-.', label='Original IFFT')
plt.legend()
plt.title("Effect of zero-insertion in frequency domain")
plt.xlabel("Real-space index")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()


1.9375 1.875
